# Leakage check and fix

In [2]:
import pandas as pd

players_fresh_stats = pd.read_csv("../data/players_fresh_stats.csv")
appearances = pd.read_csv("../data/appearances.csv")
players = pd.read_csv("../data/players.csv")
games = pd.read_csv("../data/games.csv")
appearances = pd.read_csv("../data/appearances.csv")
appearances['date'] = pd.to_datetime(appearances['date'])
appearances_dated = appearances.merge(players_fresh_stats[['player_id', 'current_date']], on='player_id', how='inner')
appearances_dated['current_date'] = pd.to_datetime(appearances_dated['current_date'])
appearances_valid = appearances_dated[appearances_dated['date'] <= appearances_dated['current_date']]

In [3]:
player_stats_fixed = appearances_valid.groupby('player_id').agg(
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    total_appearances=('appearance_id', 'count'),
    total_yellow_cards=('yellow_cards', 'sum'),
    total_red_cards=('red_cards', 'sum')
).reset_index()

player_stats_fixed['goals_per_90'] = player_stats_fixed['total_goals']*90/player_stats_fixed['total_minutes']
player_stats_fixed['assists_per_90'] = player_stats_fixed['total_assists']*90/player_stats_fixed['total_minutes']


In [4]:
comparison = players_fresh_stats[['player_id', 'name', 'total_goals']].merge(player_stats_fixed[['player_id', 'total_goals']], on='player_id', suffixes=('_leaked', '_fixed'))
comparison['diff'] = comparison['total_goals_leaked'] - comparison['total_goals_fixed']
print(comparison.sort_values('diff', ascending=False).head(10))
print((player_stats_fixed['total_minutes'] == 0).sum())

      player_id             name  total_goals_leaked  total_goals_fixed  diff
4634     132098       Harry Kane               414.0                382  32.0
1298     418560   Erling Haaland               259.0                237  22.0
2707     288230  Ousmane Dembélé               121.0                102  19.0
1320     324358    Ollie Watkins               100.0                 82  18.0
1335     325443  Viktor Gyökeres               109.0                 93  16.0
2290      46413     Ante Budimir               112.0                 97  15.0
1319     326029    Donyell Malen               119.0                104  15.0
2400     256267     Vedat Muriqi                97.0                 82  15.0
1050     626724       João Pedro                57.0                 42  15.0
4462     480692        Luis Díaz               105.0                 90  15.0
0


In [ ]:
players_fresh_final = players_fresh_stats[['player_id', 'name', 'current_value', 'current_date', 'peak_value', 'peak_date','days_since_update', 'is_stale']].merge(player_stats_fixed, on='player_id', how='inner')
players_fresh_final = players_fresh_final.merge(players[['player_id', 'date_of_birth', 'position', 'sub_position', 'foot', 'height_in_cm', 'country_of_citizenship']], on='player_id', how='inner')

players_fresh_final['date_of_birth'] = pd.to_datetime(players_fresh_final['date_of_birth'])
players_fresh_final['current_date'] = pd.to_datetime(players_fresh_final['current_date'])
players_fresh_final['age'] = (players_fresh_final['current_date'] - players_fresh_final['date_of_birth']).dt.days / 365.25

print(players_fresh_final.isna().sum())

players_fresh_final['foot'] = players_fresh_final['foot'].fillna('unknown')
players_fresh_final['height_in_cm'] = players_fresh_final.groupby('position')['height_in_cm'].transform(lambda x: x.fillna(x.median()))
print(players_fresh_final.isna().sum())
print(f"Final shape: {players_fresh_final.shape}")

After stats merge: 5585
After attributes merge: 5585
player_id                  0
name                       0
current_value              0
current_date               0
peak_value                 0
peak_date                  0
days_since_update          0
is_stale                   0
total_goals                0
total_assists              0
total_minutes              0
total_appearances          0
total_yellow_cards         0
total_red_cards            0
goals_per_90               0
assists_per_90             0
date_of_birth              0
position                   0
sub_position               0
foot                      85
height_in_cm              79
country_of_citizenship     0
age                        0
dtype: int64
player_id                 0
name                      0
current_value             0
current_date              0
peak_value                0
peak_date                 0
days_since_update         0
is_stale                  0
total_goals               0
total_assists  

In [18]:
players_fresh_final.to_csv("../data/players_fresh_final.csv", index=False)